# Ollama Model Specialization with LoRA

Single consolidated notebook for fine-tuning Ollama models using LoRA on the scholarship dataset.

**Features:**
- Automatic dependency installation
- Scholarship dataset loading
- LoRA adapter training
- Model evaluation
- Saves reusable adapters

## Instructions
1. Run all cells in sequence
2. Or configure parameters in the "Configuration" cell and run from there
3. Adapters will be saved to `./ollama/adapters/`

## 1. Setup and Auto-Install Dependencies

In [ ]:
import os
import sys
import json
import logging
from pathlib import Path
from typing import Dict, List, Any, Tuple
import warnings
import subprocess

warnings.filterwarnings('ignore')

# Auto-install dependencies
def install_dependencies():
    """Install required packages for Colab and Jupyter environments"""
    packages = [
        'torch',
        'transformers>=4.35.0',
        'peft>=0.7.1',
        'datasets>=2.14.6',
        'accelerate>=0.24.1',
        'bitsandbytes>=0.41.2',
    ]
    
    print("Installing dependencies...")
    for package in packages:
        try:
            subprocess.check_call(
                [sys.executable, '-m', 'pip', 'install', '-q', package],
                stdout=subprocess.DEVNULL,
                stderr=subprocess.DEVNULL
            )
        except:
            print(f"Could not install {package}, continuing...")
    print("Dependencies installed\n")

# Install dependencies
install_dependencies()

# Import core libraries
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModelForCausalLM, AutoTokenizer, get_linear_schedule_with_warmup
from peft import get_peft_model, LoraConfig, TaskType
import numpy as np

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

print("All imports successful")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. ScholarshipDataset Class

In [ ]:
class ScholarshipDataset(Dataset):
    """Scholarship matching dataset for fine-tuning"""
    
    def __init__(
        self,
        dataset_path: str,
        tokenizer,
        max_length: int = 512,
        num_samples: int = None
    ):
        self.tokenizer = tokenizer
        self.max_length = max_length
        
        logger.info(f"Loading dataset from {dataset_path}")
        with open(dataset_path, 'r') as f:
            self.data = json.load(f)
        
        if num_samples:
            self.data = self.data[:num_samples]
        
        logger.info(f"Loaded {len(self.data)} records")
        self.prompts = self._create_prompts()
    
    def _create_prompts(self) -> List[str]:
        """Create training prompts from scholarship data"""
        prompts = []
        
        for record in self.data:
            profile = record['profile']
            scholarship = record['scholarship']
            match_score = record['match']
            
            prompt = f"""<|scholarship_match|>
Profile: Name: {profile['full_name']}, GPA: {profile['gpa']}, Major: {profile['major']}, State: {profile['state']}, Citizenship: {profile['citizenship_status']}, Income: {profile['family_income']}, Activities: {', '.join(profile.get('extracurriculars', [])[:2])}

Scholarship: Title: {scholarship['title']}, Amount: ${scholarship['amount']}, Provider: {scholarship['provider']}, Min GPA: {scholarship['eligibility_criteria'].get('min_gpa', 'Any')}, Majors: {', '.join(scholarship['eligibility_criteria'].get('majors', ['Any'])[:2]) or 'Any'}, States: {', '.join(scholarship['eligibility_criteria'].get('states', ['Any'])[:2]) or 'Any'}

Match Score: {match_score}/100
Reasoning: {' | '.join(record['match_reasons'][:2])}
<|end_scholarship_match|>"""
            prompts.append(prompt)
        
        return prompts
    
    def __len__(self) -> int:
        return len(self.prompts)
    
    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        prompt = self.prompts[idx]
        
        encoding = self.tokenizer(
            prompt,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels': encoding['input_ids'].squeeze()
        }

print("ScholarshipDataset class loaded")

## 3. ScholarshipMatcherTrainer Class

In [ ]:
class ScholarshipMatcherTrainer:
    """Complete LoRA fine-tuning and evaluation in one class"""
    
    def __init__(
        self,
        model_name: str = 'mistral',
        output_dir: str = './ollama/adapters',
        learning_rate: float = 2e-4,
        batch_size: int = 4,
        epochs: int = 3,
        adapter_name: str = 'scholarship-matcher'
    ):
        self.model_name = model_name
        self.output_dir = Path(output_dir)
        self.learning_rate = learning_rate
        self.batch_size = batch_size
        self.epochs = epochs
        self.adapter_name = adapter_name
        
        self.output_dir.mkdir(parents=True, exist_ok=True)
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        
        logger.info(f"Using device: {self.device}")
        if self.device == 'cuda':
            logger.info(f"GPU: {torch.cuda.get_device_name(0)}")
    
    def setup_model(self):
        """Load model and configure LoRA"""
        logger.info(f"Loading base model: {self.model_name}")
        
        try:
            self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
            self.model = AutoModelForCausalLM.from_pretrained(
                self.model_name,
                torch_dtype=torch.float16 if self.device == 'cuda' else torch.float32,
                device_map='auto' if self.device == 'cuda' else None
            )
        except Exception as e:
            logger.error(f"Failed to load model: {e}")
            raise
        
        # Configure LoRA
        logger.info("Configuring LoRA adapters...")
        lora_config = LoraConfig(
            task_type=TaskType.CAUSAL_LM,
            r=8,
            lora_alpha=32,
            lora_dropout=0.05,
            bias='none',
            target_modules=['q_proj', 'v_proj'],
        )
        
        self.model = get_peft_model(self.model, lora_config)
        self.model.print_trainable_parameters()
        
        if self.device != 'cpu':
            self.model.to(self.device)
    
    def prepare_data(self, dataset_path: str, num_samples: int = None):
        """Prepare training dataset"""
        logger.info("Preparing dataset...")
        
        dataset = ScholarshipDataset(
            dataset_path,
            self.tokenizer,
            num_samples=num_samples
        )
        
        self.train_dataloader = DataLoader(
            dataset,
            batch_size=self.batch_size,
            shuffle=True
        )
        
        logger.info(f"Dataset ready with {len(dataset)} samples")
    
    def train(self):
        """Execute training loop"""
        logger.info("Starting training...")
        
        optimizer = torch.optim.AdamW(self.model.parameters(), lr=self.learning_rate)
        total_steps = len(self.train_dataloader) * self.epochs
        scheduler = get_linear_schedule_with_warmup(
            optimizer,
            num_warmup_steps=0,
            num_training_steps=total_steps
        )
        
        self.model.train()
        
        for epoch in range(self.epochs):
            logger.info(f"Epoch {epoch + 1}/{self.epochs}")
            total_loss = 0
            
            for step, batch in enumerate(self.train_dataloader):
                batch = {k: v.to(self.device) for k, v in batch.items()}
                
                outputs = self.model(**batch)
                loss = outputs.loss
                
                loss.backward()
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()
                
                total_loss += loss.item()
                
                if (step + 1) % max(1, len(self.train_dataloader) // 5) == 0:
                    avg_loss = total_loss / (step + 1)
                    logger.info(f"  Step {step + 1}/{len(self.train_dataloader)}, Loss: {avg_loss:.4f}")
            
            epoch_loss = total_loss / len(self.train_dataloader)
            logger.info(f"Epoch {epoch + 1} - Average loss: {epoch_loss:.4f}\n")
        
        logger.info("Training completed!")
    
    def save_adapters(self):
        """Save LoRA adapters and configuration"""
        adapter_path = self.output_dir / self.adapter_name
        adapter_path.mkdir(parents=True, exist_ok=True)
        
        logger.info(f"Saving adapters to {adapter_path}")
        self.model.save_pretrained(adapter_path)
        self.tokenizer.save_pretrained(adapter_path)
        
        # Save metadata
        metadata = {
            'adapter_name': self.adapter_name,
            'base_model': self.model_name,
            'task': 'scholarship_matching',
            'lora_config': {
                'r': 8,
                'lora_alpha': 32,
                'lora_dropout': 0.05,
                'target_modules': ['q_proj', 'v_proj']
            },
            'training_params': {
                'learning_rate': self.learning_rate,
                'batch_size': self.batch_size,
                'epochs': self.epochs
            }
        }
        
        with open(adapter_path / 'adapter_metadata.json', 'w') as f:
            json.dump(metadata, f, indent=2)
        
        # Create Modelfile
        modelfile = f"""FROM {self.model_name}
PARAMETER adapter_path {adapter_path}
SYSTEM You are a scholarship matching expert. Analyze student profiles and scholarships to provide accurate match scores.
PARAMETER temperature 0.7
PARAMETER top_p 0.9
PARAMETER stop "<|end_scholarship_match|>"
"""
        with open(self.output_dir / f"{self.adapter_name}.Modelfile", 'w') as f:
            f.write(modelfile)
        
        logger.info(f"Adapters saved successfully!")
        return adapter_path
    
    def evaluate(self, dataset_path: str, num_samples: int = 50) -> Dict[str, Any]:
        """Evaluate model on test samples"""
        logger.info(f"Evaluating on {num_samples} samples...")
        
        with open(dataset_path, 'r') as f:
            data = json.load(f)
        
        data = data[-num_samples:]
        
        results = []
        errors = []
        
        self.model.eval()
        
        with torch.no_grad():
            for i, record in enumerate(data):
                try:
                    profile = record['profile']
                    scholarship = record['scholarship']
                    actual_score = record['match']
                    
                    prompt = f"Match: {profile['major']} student GPA {profile['gpa']} in {profile['state']} vs {scholarship['title']} (${scholarship['amount']}) score:"
                    
                    inputs = self.tokenizer(prompt, return_tensors='pt').to(self.device)
                    outputs = self.model.generate(
                        **inputs,
                        max_new_tokens=50,
                        do_sample=True,
                        temperature=0.7,
                        top_p=0.9
                    )
                    
                    response = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
                    
                    # Extract score from response
                    try:
                        pred_score = float(''.join(c for c in response.split('\n')[-1] if c.isdigit()))
                        pred_score = min(100, max(0, pred_score))
                    except:
                        pred_score = 50
                    
                    error = abs(pred_score - actual_score)
                    errors.append(error)
                    
                    results.append({
                        'profile': profile['full_name'],
                        'predicted': pred_score,
                        'actual': actual_score,
                        'error': error
                    })
                    
                    if (i + 1) % max(1, num_samples // 5) == 0:
                        logger.info(f"  Evaluated {i + 1}/{len(data)}")
                except Exception as e:
                    logger.warning(f"  Error on sample {i}: {e}")
                    continue
        
        # Calculate metrics
        if errors:
            metrics = {
                'num_samples': len(results),
                'mean_error': np.mean(errors),
                'median_error': np.median(errors),
                'within_10': sum(1 for e in errors if e <= 10) / len(errors) * 100,
                'within_20': sum(1 for e in errors if e <= 20) / len(errors) * 100,
            }
        else:
            metrics = {'error': 'No valid predictions'}
        
        return {
            'metrics': metrics,
            'samples': results[:5]
        }
    
    def run_full_pipeline(
        self,
        dataset_path: str,
        num_train_samples: int = None,
        num_eval_samples: int = 50,
        skip_training: bool = False
    ):
        """Run complete pipeline: setup → train → evaluate → save"""
        try:
            logger.info("=" * 70)
            logger.info("Ollama Model Specialization - Notebook Edition")
            logger.info("=" * 70)
            
            if not skip_training:
                self.setup_model()
                self.prepare_data(dataset_path, num_train_samples)
                self.train()
            
            adapter_path = self.save_adapters()
            eval_results = self.evaluate(dataset_path, num_eval_samples)
            
            logger.info("\n" + "=" * 70)
            logger.info("EVALUATION RESULTS")
            logger.info("=" * 70)
            logger.info(f"Samples: {eval_results['metrics'].get('num_samples', 'N/A')}")
            logger.info(f"Mean Error: {eval_results['metrics'].get('mean_error', 'N/A'):.2f}")
            logger.info(f"Within ±10: {eval_results['metrics'].get('within_10', 'N/A'):.1f}%")
            logger.info(f"Within ±20: {eval_results['metrics'].get('within_20', 'N/A'):.1f}%")
            
            logger.info("\n" + "=" * 70)
            logger.info("PIPELINE COMPLETE")
            logger.info("=" * 70)
            logger.info(f"Adapter path: {adapter_path}")
            logger.info(f"Modelfile: {self.output_dir / f'{self.adapter_name}.Modelfile'}")
            logger.info("\nNext steps:")
            logger.info("1. ollama create scholarship-matcher -f <modelfile>")
            logger.info("2. ollama run scholarship-matcher")
            
            return {
                'adapter_path': str(adapter_path),
                'metrics': eval_results['metrics'],
                'samples': eval_results['samples']
            }
        
        except Exception as e:
            logger.error(f"Error in pipeline: {e}")
            raise

print("ScholarshipMatcherTrainer class loaded")

## 4. Configuration

In [ ]:
# ============================================================================
# CONFIGURATION
# ============================================================================

# Model and data paths
DATASET_PATH = './datasets/scholarship-profiles-dataset.json'
MODEL_NAME = 'mistral'  # Options: mistral, llama2, neural-chat
OUTPUT_DIR = './ollama/adapters'
ADAPTER_NAME = 'scholarship-matcher'

# Training parameters
LEARNING_RATE = 2e-4
BATCH_SIZE = 4
EPOCHS = 3  # Set to 1 for quick test

# Data parameters
NUM_TRAIN_SAMPLES = None  # None = use all, set to 50 for quick test
NUM_EVAL_SAMPLES = 50

# Options
SKIP_TRAINING = False  # Set to True to skip training and only evaluate

print("Configuration loaded")
print(f"  Dataset: {DATASET_PATH}")
print(f"  Model: {MODEL_NAME}")
print(f"  Output: {OUTPUT_DIR}")
print(f"  Training epochs: {EPOCHS}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Learning rate: {LEARNING_RATE}")

## 5. Quick Test Mode (Optional)

Run this cell to do a quick 2-minute test before full training.

In [ ]:
# Uncomment to enable test mode
# TEST_MODE = True

# if TEST_MODE:
#     print("TEST MODE ENABLED")
#     NUM_TRAIN_SAMPLES = 50
#     EPOCHS = 1
#     NUM_EVAL_SAMPLES = 20
#     print("Test configuration loaded")

## 6. Run Full Pipeline

In [ ]:
# Verify dataset exists
if not Path(DATASET_PATH).exists():
    print(f"Dataset not found: {DATASET_PATH}")
    print("Please generate the dataset first using: node scripts/generate-dataset.js")
else:
    print(f"Dataset found: {DATASET_PATH}")
    
    # Create trainer instance
    trainer = ScholarshipMatcherTrainer(
        model_name=MODEL_NAME,
        output_dir=OUTPUT_DIR,
        learning_rate=LEARNING_RATE,
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        adapter_name=ADAPTER_NAME
    )
    
    # Run full pipeline
    result = trainer.run_full_pipeline(
        dataset_path=DATASET_PATH,
        num_train_samples=NUM_TRAIN_SAMPLES,
        num_eval_samples=NUM_EVAL_SAMPLES,
        skip_training=SKIP_TRAINING
    )
    
    print("\n" + "="*70)
    print("Pipeline execution complete!")
    print("="*70)